# Download audio (yt-dlp)
Links from `data/podcasts.md` -> wav in `data/raw_audios/`. If YouTube asks to sign in, set `cookies_from_browser`.

In [ ]:
import re
import shutil
from pathlib import Path

from yt_dlp import YoutubeDL

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
out_dir = root / "data" / "raw_audios"
out_dir.mkdir(parents=True, exist_ok=True)

links = re.findall(r"https?://[^\s)>]+", (root / "data" / "podcasts.md").read_text(encoding="utf-8"))
print(len(links), "links")

In [ ]:
cookies_from_browser = None  # "chrome", "firefox" or "safari"


def download(link):
    opts = {
        "format": "bestaudio/best",
        "postprocessors": [{"key": "FFmpegExtractAudio", "preferredcodec": "wav"}],
        "outtmpl": str(out_dir / "%(title)s.%(ext)s"),
        "noplaylist": True,
        "remote_components": ["ejs:github"],
        "download_archive": str(out_dir / ".ytdlp_downloaded.txt"),  # skips finished ones on rerun
    }
    if node := shutil.which("node"):
        opts["js_runtimes"] = {"node": {"path": node}}
    if cookies_from_browser:
        opts["cookiesfrombrowser"] = (cookies_from_browser,)
    with YoutubeDL(opts) as ydl:
        ydl.download([link])


failed = []
for i, link in enumerate(links, 1):
    print(f"[{i}/{len(links)}] {link}")
    try:
        download(link)
    except Exception as e:
        failed.append(link)
        print("failed:", e)
print(f"done, {len(failed)} failed")